In [ ]:
# ========== 第 8 周：集成模型（Ensemble）依赖安装 ==========
# 理念：多路估价（微调 LLaMA / GPT RAG / CatBoost）+ 线性回归 stacking
# 在 Colab 安装训练与推理常用库（保持原包列表）
!pip install tqdm huggingface_hub numpy sentence-transformers datasets chromadb catboost peft torch bitsandbytes


In [ ]:
# ========== 导入：数据 / 向量库 / 模型 / 指标 ==========
# os：路径与环境
import os
# re：从生成文本抠价格
import re
# zipfile：压缩包（保留原 import）
import zipfile
# chromadb：相似商品向量检索
import chromadb
# joblib：加载已训练的 CatBoost pkl
import joblib
# numpy：数值与随机划分
import numpy as np
# pandas：堆叠特征表 DataFrame
import pandas as pd
# requests：HTTP（保留原 import）
import requests
# torch：加载量化 LLaMA
import torch
# load_dataset：从 HF Hub 拉定价数据
from datasets import load_dataset
# userdata：Colab Secrets 读 API Key
from google.colab import userdata
# HF Hub：登录与下载
from huggingface_hub import HfApi, hf_hub_download, login
# OpenAI 客户端：gpt-4o-mini RAG 估价
from openai import OpenAI
# PeftModel：加载 LoRA 微调适配器
from peft import PeftModel
# SentenceTransformer：e5 向量
from sentence_transformers import SentenceTransformer
# LinearRegression：二层 stacking
from sklearn.linear_model import LinearRegression
# r2_score / MSE：评估指标
from sklearn.metrics import r2_score, mean_squared_error
# tqdm：循环进度条
from tqdm import tqdm
# Transformers：因果 LM + tokenizer + 4bit 配置
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# CatBoostRegressor：类型提示/一致性（模型用 joblib 加载）
from catboost import CatBoostRegressor


In [4]:
# ========== 挂载 Google Drive：读 Chroma / CatBoost 权重 ==========
# Colab Drive 工具
from google.colab import drive
# 挂载到 /content/drive
drive.mount("/content/drive")


Mounted at /content/drive


In [5]:
# ========== 密钥：OpenAI + Hugging Face（Colab Secrets） ==========
# 从 Colab userdata 取 OPENAI_API_KEY
openai_api_key = userdata.get("OPENAI_API_KEY")
# 创建 OpenAI 客户端
openai = OpenAI(api_key=openai_api_key)

# HF token
hf_token = userdata.get("HF_TOKEN")
# 登录；允许写 git credential（原参数保留）
login(hf_token, add_to_git_credential=True)


In [6]:
# 配置：HF 用户名（数据集命名空间）
# 配置
# 你的 Hugging Face 用户名，用于拼 DATASET_NAME
HF_USER = "qshaikh"


In [ ]:
# ========== 加载定价数据集的 test split ==========
# 数据集 ID：{user}/pricer-data
DATASET_NAME = f"{HF_USER}/pricer-data"
# 从 Hub 拉取
dataset = load_dataset(DATASET_NAME)
# 只要 test 部分做评估/堆叠训练
test = dataset["test"]


In [ ]:
# ========== description：从 item 文本抽出商品段落 ==========
# 输入一条 dataset item，返回 e5 风格 passage 文本
def description(item):
    # 去掉训练时常用的提问前缀
    text = item["text"].replace(
        # 前缀字符串保持原样
        "How much does this cost to the nearest dollar?\n\n", ""
    )
    # 去掉末尾 Price is $... 监督信号，只留描述
    text = text.split("\n\nPrice is $")[0]
    # 加 passage: 前缀，匹配 e5 双编码器用法
    return f"passage: {text}"


In [ ]:
# ========== Chroma：从 Drive 加载已建好的价格向量库 ==========
# Drive 上的 Chroma 持久化路径
CHROMA_PATH = "/content/drive/MyDrive/chroma"
# 集合名
COLLECTION_NAME = "price_items"

# 提示正在加载的路径
print(f"Attempting to load ChromaDB from: {CHROMA_PATH}")

# 打开持久化客户端
client = chromadb.PersistentClient(path=CHROMA_PATH)
# 获取或创建集合
collection = client.get_or_create_collection(name=COLLECTION_NAME)

# 加载成功提示
print(f"Successfully loaded ChromaDB collection '{COLLECTION_NAME}'.")


In [ ]:
# ========== Embedding 模型：intfloat/e5-small-v2（GPU） ==========
# device=cuda：在 GPU 上编码；模型 id 保持原样
embedding_model = SentenceTransformer("intfloat/e5-small-v2", device="cuda")


In [ ]:
# ========== 加载 4bit 基座 LLaMA + LoRA 微调定价适配器 ==========
# 基座模型 id
BASE_MODEL = "meta-llama/Llama-3.1-8B"
# PEFT/LoRA 适配器仓库
FINETUNED_MODEL = "ed-donner/pricer-2024-09-13_13.04.39"
# 固定 revision，保证可复现
REVISION = "e8d637df551603dc86cd7a1598a8f44af4d7ae36"

# BitsAndBytes 4bit 量化配置，省显存
quant_config = BitsAndBytesConfig(
    # 启用 4bit 权重
    load_in_4bit=True,
    # 双重量化进一步省内存
    bnb_4bit_use_double_quant=True,
    # 计算 dtype：bfloat16
    bnb_4bit_compute_dtype=torch.bfloat16,
    # 量化类型 nf4
    bnb_4bit_quant_type="nf4",
)

# 加载 tokenizer；trust_remote_code 保持原样
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# pad 用 eos，避免缺 pad_token
tokenizer.pad_token = tokenizer.eos_token
# 右侧 padding（生成任务常见设置）
tokenizer.padding_side = "right"

# 加载量化基座；device_map=auto 自动切设备
base_model = AutoModelForCausalLM.from_pretrained(
    # 传入量化配置
    BASE_MODEL, quantization_config=quant_config, device_map="auto"
)

# 把 LoRA 适配器挂到基座上
fine_tuned_model = PeftModel.from_pretrained(
    # 指定仓库与 revision
    base_model, FINETUNED_MODEL, revision=REVISION
)

# 生成配置的 pad_token_id 与 tokenizer 对齐
fine_tuned_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 打印显存占用（MB）
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")


In [ ]:
# ========== 加载已训练好的 CatBoost 回归器（joblib pkl） ==========
#Cat Boost 训练模型
# Drive 上的模型路径
catboost_model_path = "/content/drive/MyDrive/catboost_model.pkl"
# 反序列化回归器
catboost_model = joblib.load(catboost_model_path)
# 确认加载成功
print(f"Successfully loaded CatBoost model from {catboost_model_path}")


In [24]:
# ========== extract_tagged_price：从生成文本里抠 Price is $ 后的数字 ==========
# 容错解析模型输出
def extract_tagged_price(output: str):
    # 尝试按标记切开
    try:
        # 取标记后半段并去逗号
        contents = output.split("Price is $")[1].replace(",", "")
        # 正则找数字
        match = re.search(r"[-+]?\d*\.\d+|\d+", contents)
        # 有则 float，无则 0.0
        return float(match.group()) if match else 0.0
    # 任意异常 → 0.0，避免整批评估中断
    except Exception:
        # 失败默认价
        return 0.0


In [25]:
# ========== 路径 A：微调 LLaMA 生成价格 ==========
# 输入商品描述，返回点估计价格
def ft_llama_price(description: str):
    # 构造与训练一致的 prompt（英文模板保留）
    prompt = (
        # 提问到 Price is $ 为止，让模型续写数字
        f"How much does this cost to the nearest dollar?\n\n{description}\n\nPrice is $"
    )
    # tokenize 并放到 GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 贪婪短生成：最多 5 个新 token
    outputs = fine_tuned_model.generate(
        # 解包 inputs；只要 1 条序列
        **inputs, max_new_tokens=5, num_return_sequences=1
    )

    # 解码整段文本
    result = tokenizer.decode(outputs[0])
    # 用标记解析器抽价格
    price = extract_tagged_price(result)
    # 返回 float 价格
    return price


In [26]:
# ========== 路径 B：e5 向量 → CatBoost 回归价 ==========
# 描述 → embedding → 回归
def catboost_price(description: str):
    # normalize_embeddings=True 与训练时一致
    vector = embedding_model.encode([description], normalize_embeddings=True)[0]
    # predict 后取标量
    pred = catboost_model.predict([vector])[0]
    # 价格截断到 >=0，保留 2 位小数
    return round(float(max(0, pred)), 2)


In [27]:
# ========== 路径 C：GPT-4o-mini + Chroma RAG 估价 ==========
# 对一条 item 做相似品检索，再让 GPT 只输出数字
def gpt4o_price(item):
    # 内部：编码文本
    def get_embedding(text):
        # 返回 e5 向量（可能是 2D）
        return embedding_model.encode([text], normalize_embeddings=True)

    # Chroma 近邻检索
    def find_similars(text):
        # query 顶层
        results = collection.query(
            # 向量转 list[float]；取 5 个邻居
            query_embeddings=get_embedding(text).astype(float).tolist(), n_results=5
        )
        # 相似商品文档
        docs = results["documents"][0]
        # 邻居 metadata 里的 price
        prices = [m["price"] for m in results["metadatas"][0]]
        # 返回 docs + prices
        return docs, prices

    # 把邻居格式化成英文上下文（prompt 保留）
    def format_context(similars, prices):
        # 引导语
        context = (
            "To provide some context, here are similar products and their prices:\n\n"
        # 逐条追加 Product / Price
        )
        # 格式化一行价格
        for sim, price in zip(similars, prices):
            # 返回上下文字符串
            context += f"Product:\n{sim}\nPrice is ${price:.2f}\n\n"
        return context

    # 拼 system / user / assistant 前缀消息
    def build_messages(description, similars, prices):
        # system：定价专家，只许输出数字（英文保留）
        system_message = (
            "You are a pricing expert. "
            "Given a product description and a few similar products with their prices, "
            "estimate the most likely price. "
            "Respond ONLY with a number, no words."
        )
        # 相似品上下文
        context = format_context(similars, prices)
        # user：待估商品 + 上下文
        user_prompt = (
            "Estimate the price for the following product:\n\n"
            + description
            + "\n\n"
            + context
        )
        # 三条 message；assistant 预填 Price is $ 引导续写
        return [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": "Price is $"},
        ]

    # 对 item 抽描述并检索
    docs, prices = find_similars(description(item))
    # 构建 messages
    messages = build_messages(description(item), docs, prices)
    # 调用 gpt-4o-mini；seed/max_tokens 保持原样
    response = openai.chat.completions.create(
        # 模型与采样参数不改
        model="gpt-4o-mini", messages=messages, seed=42, max_tokens=5
    )
    # 取回复文本
    reply = response.choices[0].message.content
    # 正则抽数字；失败时 .group() 可能异常——保持原逻辑
    return float(
        # 去 $ 与逗号后再搜数字
        re.search(r"[-+]?\d*\.\d+|\d+", reply.replace("$", "").replace(",", "")).group()
        or 0
    )


In [ ]:
# ========== 划分索引：从 test 里再切 80/20，并子采样 ==========
# 提示开始划分
print("Splitting entire dataset...")
# 固定种子，结果可复现
np.random.seed(42)
# 所有 test 下标
all_indices = list(range(len(test)))
# 原地打乱
np.random.shuffle(all_indices)

# 80% 当作 stacking 训练集
train_split_size = int(0.8 * len(all_indices))
# 前 80%
train_indices = all_indices[:train_split_size]  # 80%
# 后 20%
test_indices = all_indices[train_split_size:]  # 20%

# 再截断：最多 250 条训练
train_indices = train_indices[:250]
# 最多 50 条测试（控 API/算力成本）
test_indices = test_indices[:50]


In [ ]:
# ========== 在 train_indices 上跑三路模型，收集 stacking 特征 ==========
# 微调 LLaMA 预测列表
ft_llama_preds_train = []
# GPT-4o-mini 预测列表
gpt4omini_preds_train = []
# CatBoost 预测列表
catboost_preds_train = []
# 真实价格列表
true_prices_train = []

# 逐条索引推理（可能较慢/耗 API）
for i in tqdm(train_indices):
    # 注意：这里用的是 test 子集的元素
    item = test[i]
    # 清洗后的描述文本
    text = description(item)
    # 真值价
    true_prices_train.append(item["price"])
    # 路径 A
    ft_llama_preds_train.append(ft_llama_price(text))
    # 路径 C（内部会再 description）
    gpt4omini_preds_train.append(gpt4o_price(item))
    # 路径 B
    catboost_preds_train.append(catboost_price(text))


In [ ]:
# ========== 目检：打印训练子集上的真值与三路预测 ==========
# 真值
print("True Prices:", true_prices_train)
# FT-LLaMA
print("FT-LLaMA Predictions:", ft_llama_preds_train)
# GPT-4o-mini
print("GPT-4o-mini Predictions:", gpt4omini_preds_train)
# CatBoost
print("CatBoost Predictions:", catboost_preds_train)


In [ ]:
# ========== 构造 stacking 特征：三路预测 + Max + Mean ==========
# 逐样本取三路最大值
maxes_train = [
    # max(a,b,c)
    max(a, b, c)
    # 与三路预测 zip
    for a, b, c in zip(ft_llama_preds_train, gpt4omini_preds_train, catboost_preds_train)
]
# 逐样本取三路均值
means_train = [
    # np.mean
    np.mean([a, b, c])
    # 与三路预测 zip
    for a, b, c in zip(ft_llama_preds_train, gpt4omini_preds_train, catboost_preds_train)
]

# 特征矩阵 X_train
X_train = pd.DataFrame(
    # 列：三个基学习器 + Max + Mean
    {
        "FT_LLaMA": ft_llama_preds_train,
        "GPT4oMini": gpt4omini_preds_train,
        "CatBoost": catboost_preds_train,
        "Max": maxes_train,
        "Mean": means_train,
    }
# 标签：真实价格
)

y_train = pd.Series(true_prices_train)


In [ ]:
# ========== 训练二层 LinearRegression，并打印系数 ==========
# 固定种子（本格主要影响其它随机性）
np.random.seed(42)
# 线性回归元模型
lr = LinearRegression()
# 用 stacking 特征拟合真值
lr.fit(X_train, y_train)

# 特征名列表
feature_columns = X_train.columns.tolist()
# 逐特征打印权重，看谁更被信任
for feature, coef in zip(feature_columns, lr.coef_):
    # 系数保留 2 位
    print(f"{feature}: {coef:.2f}")
# 截距
print(f"Intercept={lr.intercept_:.2f}")


In [ ]:
# ========== 在 test_indices 上同样生成三路特征，供评估 ==========
# 测试集：FT-LLaMA 预测
ft_llama_preds_test = []
# 测试集：GPT
gpt4omini_preds_test = []
# 测试集：CatBoost
catboost_preds_test = []
# 测试集真值
true_prices_test = []

# 提示开始处理 50 条测试
print("Processing TEST data (50 items)...")
# 循环测试索引
for i in tqdm(test_indices):
    # 取 item
    item = test[i]
    # 描述
    text = description(item)
    # 真值
    true_prices_test.append(item["price"])
    # 三路推理
    ft_llama_preds_test.append(ft_llama_price(text))
    # GPT RAG
    gpt4omini_preds_test.append(gpt4o_price(item))
    # CatBoost
    catboost_preds_test.append(catboost_price(text))

# 测试集 Max 特征
maxes_test = [
    # max
    max(a, b, c)
    # zip 三路
    for a, b, c in zip(ft_llama_preds_test, gpt4omini_preds_test, catboost_preds_test)
]
# 测试集 Mean 特征
means_test = [
    # mean
    np.mean([a, b, c])
    # zip 三路
    for a, b, c in zip(ft_llama_preds_test, gpt4omini_preds_test, catboost_preds_test)
]

# 测试特征表，列名与训练一致
X_test = pd.DataFrame(
    # 五列特征
    {
        "FT_LLaMA": ft_llama_preds_test,
        "GPT4oMini": gpt4omini_preds_test,
        "CatBoost": catboost_preds_test,
        "Max": maxes_test,
        "Mean": means_test,
    }
# 测试标签
)

y_test = pd.Series(true_prices_test)


In [ ]:
# ========== 评估集成：R² / RMSE / MAPE ==========
# 开始评估提示
print("Evaluating model...")
# 元模型在 X_test 上预测
y_pred = lr.predict(X_test)
# 决定系数 R²
r2 = r2_score(y_test, y_pred)
# 打印 R²
print(f"R² score: {r2:.4f}")

# 均方根误差 RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# 打印 RMSE
print(f"RMSE: {rmse:.2f}")

# 平均绝对百分比误差 MAPE（%）
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
# 打印 MAPE
print(f"MAPE: {mape:.2f}%")
